# 🤖 Виртуальный консультант ювелирного магазина с использованием RAG

Краткое описание проекта.

Проект представляет собой AI-консультанта для ювелирного магазина.

Система использует технологию RAG (Retrieval-Augmented Generation):
- выполняет поиск информации в базе знаний;
- ищет подходящие товары в каталоге;
- формирует контекст;
- генерирует ответ с помощью языковой модели Llama.

## Глава 1. Подготовка к работе

### 1.1. Установка зависимостей

На данном этапе устанавливаются библиотеки, необходимые для работы с:

- ChromaDB (векторная база данных);
- FAISS (быстрый поиск по векторам);
- python-dotenv (загрузка переменных окружения из файла `.env`).

In [1]:
!pip install -q faiss-cpu chromadb python-dotenv groq

### 1.2. Импорт библиотек

Импортируем и проверяем, что необходимые библиотеки установлены и доступны в текущей среде выполнения Colab:



*   chromadb → наша векторная база данных;
*   faiss → библиотека поиска по векторам (потом можем использовать для сравнения);
*   torch → нужен для работы с нейросетевыми моделями;
*   transformers → загрузка и запуск языковых моделей;
*   sentence_transformers → создание эмбеддингов текста.


In [2]:
import chromadb
import faiss
import torch
import transformers
import sentence_transformers
import groq
from groq import Groq

print("✓ Chroma:", chromadb.__version__)
print("✓ FAISS:", faiss.__version__)
print("✓ Torch:", torch.__version__)
print("✓ Transformers:", transformers.__version__)
print("✓ SentenceTransformers:", sentence_transformers.__version__)
print("✓ Groq:", groq.__version__)

✓ Chroma: 1.5.9
✓ FAISS: 1.14.3
✓ Torch: 2.11.0+cpu
✓ Transformers: 5.13.1
✓ SentenceTransformers: 5.6.0
✓ Groq: 1.6.0


### 1.3. Настройка переменных окружения

Для работы с внешними API используются переменные окружения.

В файле `.env` хранятся ключи доступа к сервисам Unstructured и Groq.

При публикации проекта реальные ключи не размещаются в репозитории. Вместо них используется файл `.env.example`.

In [3]:
%%writefile .env

UNSTRUCTURED_API_KEY=your_unstructured_api_key
UNSTRUCTURED_API_URL=https://platform-api.transform.unstructured.io/api/v1

GROQ_API_KEY=your_groq_api_key

Overwriting .env


### 1.4. Проверка подключения к API

На этом этапе проверяется корректность настройки внешних сервисов, которые используются в проекте.

Переменные окружения загружаются из файла `.env`, после чего проверяется наличие необходимых параметров:
- ключа Unstructured API для обработки документов;
- ключа Groq API для доступа к языковой модели Llama.

После загрузки настроек создаётся клиент Groq и выполняется тестовый запрос к языковой модели. Это позволяет убедиться, что подключение к API успешно и модель готова использоваться в дальнейшем процессе генерации ответов.

In [4]:
from dotenv import load_dotenv
import os

load_dotenv()

print("Unstructured API Key найден:", bool(os.getenv("UNSTRUCTURED_API_KEY")))
print("Unstructured API URL настроен:", bool(os.getenv("UNSTRUCTURED_API_URL")))
print("Groq API Key найден:", bool(os.getenv("GROQ_API_KEY")))

Unstructured API Key найден: True
Unstructured API URL настроен: True
Groq API Key найден: True


In [5]:
client_llama = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

print("✓ Groq клиент создан")

✓ Groq клиент создан


In [6]:
response = client_llama.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": "Привет! Представься кратко."
        }
    ]
)

print(response.choices[0].message.content)

Привет! Я - языковая модель, способная отвечать на вопросы и беседовать.


## Глава 2. Создание базы знаний

На этом этапе создается база знаний виртуального консультанта. Подготавливаются текстовые документы с информацией о ювелирных изделиях, создается векторная база ChromaDB, выполняется преобразование документов в эмбеддинги и проверяется семантический поиск.

### 2.1. Создание векторного хранилища ChromaDB

ChromaDB используется для хранения документов и их векторных представлений.

PersistentClient создаёт постоянное хранилище, которое сохраняется в указанной папке и может быть повторно загружено после перезапуска среды выполнения.

In [7]:
import os

os.makedirs("chroma", exist_ok=True)

print("Папка существует:", os.path.exists("chroma"))

Папка существует: True


In [8]:
client = chromadb.PersistentClient(path="./chroma")
print("Chroma работает!")

Chroma работает!


### 2.2. Проверка создания эмбеддингов

Для поиска по смыслу текст необходимо преобразовать в числовой вектор.

SentenceTransformer использует нейросетевую модель, которая преобразует текст в набор чисел. Полученный вектор позволяет сравнивать смысловую близость разных текстов.

In [9]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embedding = model.encode("Золотые украшения требуют бережного ухода.")

print(type(embedding))
print(len(embedding))
print(embedding[:10])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

<class 'numpy.ndarray'>
384
[ 0.04625959  0.08351605  0.04384207  0.00151736 -0.06057939 -0.01361857
  0.10196688  0.05845584 -0.09112766 -0.03827967]


### 2.3. Подготовка базы знаний

Для RAG-системы необходим внешний источник знаний, из которого модель сможет извлекать информацию.

В качестве базы знаний создадим набор текстовых документов о ювелирных изделиях:
- уход за украшениями;
- особенности драгоценных металлов;
- информация о драгоценных камнях;
- рекомендации по выбору украшений.

Эти документы будут преобразованы в векторное представление и сохранены в векторной базе данных.

In [10]:
os.makedirs("data", exist_ok=True)

print(os.listdir())

['.config', 'chroma', 'data', '.env', 'sample_data']


In [11]:
%%writefile data/care.txt

# Уход за ювелирными изделиями

## Общие правила ухода

Ювелирные украшения рассчитаны на длительное использование, однако их внешний вид во многом зависит от правильного ухода. Даже самые прочные драгоценные металлы и камни постепенно теряют блеск при постоянном контакте с косметикой, бытовой химией, потом и загрязнениями окружающей среды.

Регулярная очистка и правильное хранение позволяют сохранить первоначальный вид изделия на долгие годы.

Перед выполнением домашних работ, посещением бассейна, сауны, спортивных тренировок или использованием агрессивных чистящих средств рекомендуется снимать украшения. Это снижает риск механических повреждений и воздействия химических веществ на металл и вставки.

## Уход за золотыми украшениями

Золотые изделия рекомендуется очищать по мере загрязнения, обычно один раз в несколько месяцев.

Для домашней чистки подходит теплая вода с небольшим количеством жидкого мыла. Украшение можно оставить в растворе на несколько минут, затем аккуратно очистить мягкой щеткой или тканью, промыть чистой водой и полностью высушить.

Не рекомендуется использовать:
- жесткие щетки;
- металлические губки;
- абразивные порошки;
- агрессивные химические вещества.

Такие средства могут оставить царапины на поверхности изделия и ухудшить внешний вид полированных деталей.

Если украшение покрыто родием, чрезмерная механическая чистка может постепенно удалить защитное покрытие. В таком случае рекомендуется профессиональная полировка и повторное родирование в ювелирной мастерской.

## Уход за серебряными украшениями

Серебро 925 пробы со временем может темнеть. Это естественный процесс, связанный с реакцией металла на соединения серы, содержащиеся в воздухе, косметике и некоторых бытовых средствах.

Потемнение серебра не является признаком низкого качества изделия.

Для очистки серебра рекомендуется использовать специальные:
- салфетки для серебра;
- пасты;
- растворы для ювелирных изделий.

После обработки изделие следует тщательно протереть мягкой тканью.

Не рекомендуется очищать серебро содой, зубным порошком или другими абразивными средствами. Они могут оставить на поверхности микроцарапины.

## Украшения с драгоценными камнями

Украшения с бриллиантами, сапфирами и рубинами достаточно устойчивы к повседневному использованию, однако крепления камней требуют регулярного осмотра.

Если камень начинает шататься или появляется люфт, изделие необходимо показать ювелиру.

Изумруды, жемчуг, опалы и некоторые другие минералы более чувствительны к:
- перепадам температуры;
- химическим веществам;
- механическим воздействиям.

Такие украшения рекомендуется снимать перед использованием косметики, духов и средств бытовой химии.

Жемчуг особенно чувствителен к пересыханию. После ношения его желательно аккуратно протирать мягкой тканью, удаляя остатки косметики и кожного жира.

## Правильное хранение украшений

Лучше всего хранить каждое украшение отдельно в:
- индивидуальном футляре;
- мягком мешочке;
- отдельной ячейке шкатулки.

Это предотвращает появление царапин, особенно если изделия содержат твердые драгоценные камни.

Помещение для хранения должно быть сухим, без резких перепадов температуры и повышенной влажности.

Длительное воздействие прямых солнечных лучей нежелательно для некоторых минералов.

## Чего нельзя делать

Не рекомендуется носить украшения во время:
- занятий спортом;
- ремонта;
- садовых работ;
- работы с химическими веществами.

Не следует подвергать изделия сильным ударам, особенно если они имеют крупные вставки или тонкие декоративные элементы.

Не рекомендуется использовать ультразвуковые очистители без консультации с ювелиром. Некоторые натуральные камни могут иметь внутренние трещины или включения, которые способны повредиться во время такой очистки.

## Когда обращаться к ювелиру

Профессиональная проверка украшений рекомендуется не реже одного раза в год.

Специалист может проверить:
- состояние закрепки камней;
- износ замков;
- целостность цепочек;
- наличие микроповреждений.

Обратиться в ювелирную мастерскую следует при:
- появлении трещин;
- сильной деформации изделия;
- выпадении камня;
- износе родиевого покрытия;
- повреждении застежек.

Своевременный ремонт позволяет избежать более серьезных повреждений и сохранить украшение в хорошем состоянии.

Overwriting data/care.txt


In [12]:
%%writefile data/gold_and_silver.txt
# Золото и серебро в ювелирных изделиях

## Что такое проба

Проба показывает содержание драгоценного металла в сплаве. Чистые металлы часто слишком мягкие для изготовления украшений, поэтому в них добавляют другие элементы для повышения прочности и изменения цвета.

В России чаще всего используются метрические пробы. Например, проба 585 означает, что в составе сплава содержится 58,5% чистого золота, а остальные 41,5% приходится на другие металлы.

## Золото 375, 585, 750

Золото 375 пробы содержит 37,5% чистого золота. Такие изделия отличаются высокой прочностью, но имеют менее выраженный золотой цвет из-за большего количества примесей.

Золото 585 пробы является самым распространенным материалом для ювелирных украшений. Оно сочетает в себе прочность, долговечность и привлекательный внешний вид.

Золото 750 пробы содержит 75% чистого золота. Оно имеет более насыщенный цвет и считается более ценным, но является мягче, поэтому требует более бережного обращения.

## Почему чистое золото почти не используют

Чистое золото высокой пробы является очень мягким металлом. Изделия из такого материала легко царапаются, деформируются и теряют форму при ежедневной носке.

Поэтому в ювелирном производстве используют сплавы золота с другими металлами: серебром, медью, палладием и другими элементами.

## Белое золото

Белое золото получают путем добавления в золотой сплав металлов белого цвета, например палладия или никеля.

Часто изделия из белого золота покрывают родием. Такое покрытие делает поверхность более яркой, защищает металл и придает характерный холодный блеск.

## Родирование

Родирование — это нанесение тонкого слоя родия на поверхность ювелирного изделия.

Родиевое покрытие:
- повышает блеск украшения;
- защищает поверхность от царапин;
- делает цвет белого золота более выразительным.

Со временем покрытие может стираться, поэтому изделию может потребоваться повторное родирование у ювелира.

## Серебро 925

Серебро 925 пробы содержит 92,5% чистого серебра и 7,5% других металлов.

Такой сплав называют стерлинговым серебром. Он достаточно прочный для изготовления украшений и сохраняет характерный цвет благородного металла.

## Почему серебро темнеет

Потемнение серебра является естественным процессом. Оно связано с реакцией металла с окружающей средой, особенно с соединениями серы.

На скорость потемнения могут влиять:
- влажность воздуха;
- контакт с косметикой;
- особенности кожи человека;
- неправильное хранение.

Потемневшее серебро можно аккуратно очистить специальными средствами для ювелирных изделий.

## Аллергия на ювелирные сплавы

Аллергическая реакция чаще всего возникает не на само золото или серебро, а на дополнительные металлы в составе сплава.

Например, некоторые люди чувствительны к никелю.

При склонности к аллергии рекомендуется выбирать украшения из качественных сплавов, изделий с высоким содержанием драгоценного металла или использовать защитные покрытия.

## Как отличить настоящее изделие

Для проверки подлинности украшения следует обратить внимание на:
- наличие клейма и пробы;
- качество изготовления;
- документы и сертификаты;
- репутацию продавца.

Самостоятельные проверки дома не всегда дают точный результат. Для надежной оценки лучше обратиться к специалисту.

Overwriting data/gold_and_silver.txt


In [13]:
%%writefile data/gemstones.txt
# Драгоценные камни и особенности ухода

## Что такое драгоценные камни

Драгоценные камни — это природные минералы, которые ценятся благодаря красоте, редкости, прочности и другим характеристикам.

При оценке камней учитывают:
- цвет;
- чистоту;
- размер;
- качество обработки;
- происхождение.

## Бриллиант

Бриллиант — это обработанный алмаз, которому путем огранки придают особую форму и способность отражать свет.

Главные характеристики бриллианта:
- масса в каратах;
- цвет;
- чистота;
- качество огранки.

Бриллианты отличаются высокой твердостью, но требуют аккуратного обращения, так как сильные удары могут повредить камень.

## Сапфир

Сапфир относится к разновидности корунда. Наиболее известны синие сапфиры, но камень может иметь разные оттенки.

Сапфиры ценятся за:
- высокую прочность;
- насыщенный цвет;
- устойчивость к повседневной носке.

## Изумруд

Изумруд является разновидностью берилла и известен своим зеленым цветом.

Изумруды часто имеют природные включения, которые не всегда считаются недостатком.

Этот камень требует бережного обращения, так как может быть более чувствительным к ударам и резким перепадам температуры.

## Рубин

Рубин также относится к корундам. Его характерный цвет связан с наличием хрома в составе минерала.

Рубины ценятся за:
- яркий красный цвет;
- редкость качественных экземпляров;
- высокую прочность.

## Жемчуг

Жемчуг отличается от большинства драгоценных камней тем, что имеет органическое происхождение.

Он образуется внутри раковин моллюсков.

Жемчуг чувствителен к:
- косметике;
- парфюму;
- кислотам;
- сухому воздуху.

## Твердость по Моосу

Шкала Мооса показывает устойчивость минерала к царапинам.

Некоторые значения:
- алмаз — 10;
- сапфир и рубин — 9;
- кварц — 7.

Чем выше показатель твердости, тем устойчивее камень к появлению царапин.

## Как ухаживать за камнями

Общие рекомендации:
- снимать украшения перед занятиями спортом и домашними работами;
- избегать контакта с химическими средствами;
- очищать украшения мягкой тканью;
- использовать специальные средства для ухода.

Некоторые камни требуют индивидуального ухода, поэтому перед чисткой важно учитывать свойства конкретного минерала.

## Как хранить украшения с камнями

Украшения рекомендуется хранить отдельно друг от друга, чтобы избежать царапин.

Лучше использовать:
- индивидуальные мешочки;
- специальные коробки с мягкой отделкой;
- сухое место без прямых солнечных лучей.

Overwriting data/gemstones.txt


In [14]:
%%writefile data/buying_guide.txt
# Руководство по выбору ювелирных украшений

## Как выбрать украшение

При выборе украшения важно учитывать:
- назначение изделия;
- стиль человека;
- материал;
- размер;
- бюджет.

Украшение должно подходить не только внешне, но и соответствовать образу жизни владельца.

## Как выбрать подарок

При выборе подарка стоит учитывать предпочтения человека:
- любимый цвет металла;
- стиль одежды;
- наличие других украшений;
- повод для подарка.

Универсальными вариантами часто являются классические серьги, подвески и браслеты.

## Украшения для повседневной носки

Для ежедневного использования обычно выбирают практичные украшения:
- небольшие серьги;
- тонкие цепочки;
- лаконичные кольца;
- минималистичные браслеты.

Такие изделия должны быть удобными и устойчивыми к регулярному использованию.

## Для делового стиля

Для рабочего образа подходят сдержанные украшения.

Хорошим выбором могут быть:
- небольшие серьги;
- классические часы;
- тонкие цепочки;
- украшения без чрезмерного количества декоративных элементов.

## Для торжеств

Для особых мероприятий можно выбирать более выразительные украшения:
- колье;
- серьги с драгоценными камнями;
- украшения с необычным дизайном.

Важно, чтобы украшение сочеталось с одеждой и общим стилем образа.

## Кольцо для помолвки

При выборе кольца для помолвки часто обращают внимание на:
- качество камня;
- надежность закрепки;
- удобство ежедневной носки;
- предпочтения будущей владелицы.

Классическим вариантом считается кольцо с бриллиантом, но выбор зависит от индивидуального вкуса.

## Как определить размер кольца

Размер кольца можно определить несколькими способами:
- измерить внутренний диаметр подходящего кольца;
- использовать специальную измерительную ленту;
- обратиться в ювелирный магазин.

Измерения лучше проводить вечером, когда размер пальца наиболее стабилен.

## Как подобрать серьги

При выборе серег учитывают:
- форму лица;
- длину волос;
- стиль одежды;
- удобство застежки.

Для ежедневного использования часто выбирают легкие модели, а для мероприятий подходят более заметные варианты.

## Как подобрать украшение по бюджету

Стоимость украшения зависит от:
- материала;
- пробы металла;
- наличия и характеристик камней;
- сложности изготовления.

Даже при ограниченном бюджете можно выбрать качественное украшение, если правильно подобрать материал и дизайн.

## Распространенные ошибки покупателей

Частые ошибки:
- выбор украшения только по внешнему виду без учета удобства;
- отсутствие проверки размера;
- игнорирование качества материала;
- покупка без информации о пробе и характеристиках изделия.

Перед покупкой рекомендуется изучить характеристики украшения и выбирать надежного продавца.

Overwriting data/buying_guide.txt


In [15]:
files = os.listdir("data")

print(files)

['gemstones.txt', 'gold_and_silver.txt', 'care.txt', 'catalog.json', 'buying_guide.txt']


In [16]:
for file in files:
    print("\n---", file, "---")

    with open(f"data/{file}", encoding="utf-8") as f:
        text = f.read()
        print(text[:300])


--- gemstones.txt ---
# Драгоценные камни и особенности ухода

## Что такое драгоценные камни

Драгоценные камни — это природные минералы, которые ценятся благодаря красоте, редкости, прочности и другим характеристикам.

При оценке камней учитывают:
- цвет;
- чистоту;
- размер;
- качество обработки;
- происхождение.

## 

--- gold_and_silver.txt ---
# Золото и серебро в ювелирных изделиях

## Что такое проба

Проба показывает содержание драгоценного металла в сплаве. Чистые металлы часто слишком мягкие для изготовления украшений, поэтому в них добавляют другие элементы для повышения прочности и изменения цвета.

В России чаще всего используются

--- care.txt ---

# Уход за ювелирными изделиями

## Общие правила ухода

Ювелирные украшения рассчитаны на длительное использование, однако их внешний вид во многом зависит от правильного ухода. Даже самые прочные драгоценные металлы и камни постепенно теряют блеск при постоянном контакте с косметикой, бытовой хими

--- catalog.json ---
[
   

### 2.4. Загрузка документов в LangChain

На данном этапе текстовые файлы из базы знаний преобразуются в объекты Document библиотеки LangChain.

Каждый объект содержит:
- page_content — основной текст документа;
- metadata — дополнительную информацию о документе.

Формат Document необходим для дальнейшего разбиения текста на фрагменты и создания эмбеддингов.

In [17]:
from langchain_core.documents import Document

In [18]:
documents = []

for filename in os.listdir("data"):
    if filename.endswith(".txt"):

        path = os.path.join("data", filename)

        with open(path, encoding="utf-8") as f:
            text = f.read()

        documents.append(
            Document(
                page_content=text,
                metadata={
                    "source": filename
                }
            )
        )

print(f"Загружено документов: {len(documents)}")

Загружено документов: 4


In [19]:
print(documents[0].metadata)
print(documents[1].metadata)
print(documents[2].metadata)
print(documents[3].metadata)

{'source': 'gemstones.txt'}
{'source': 'gold_and_silver.txt'}
{'source': 'care.txt'}
{'source': 'buying_guide.txt'}


### 2.5. Создание коллекции ChromaDB

На данном этапе создаётся коллекция, в которой будут храниться
подготовленные документы базы знаний.

Каждый документ будет представлен:
- уникальным идентификатором;
- текстовым содержимым;
- метаданными об исходном файле.

В дальнейшем ChromaDB будет использовать эти данные
для поиска релевантной информации по запросам пользователя.

In [20]:
collection_name = "jewelry_knowledge"

client = chromadb.PersistentClient(
    path="./chroma"
)

# удаляем старую коллекцию, если она существует
if collection_name in [c.name for c in client.list_collections()]:
    client.delete_collection(collection_name)

collection = client.create_collection(
    name=collection_name
)

print("Коллекция создана:", collection_name)

Коллекция создана: jewelry_knowledge


In [21]:
ids = [str(i) for i in range(len(documents))]

texts = [
    doc.page_content
    for doc in documents
]

metadatas = [
    doc.metadata
    for doc in documents
]


collection.add(
    ids=ids,
    documents=texts,
    metadatas=metadatas
)

print("Документы добавлены в Chroma.")

Документы добавлены в Chroma.


Так же после загрузки проверим количество документов в ChromaDB:

In [22]:
collection.count()

4

### 2.6. Тестирование семантического поиска

Проверим, насколько хорошо ChromaDB извлекает релевантные документы.

В качестве тестов используем вопросы, которые могут задавать покупатели ювелирного магазина.

In [23]:
result = collection.query(
    query_texts=[
        "Как правильно ухаживать за золотым кольцом?"
    ],
    n_results=1
)

for doc in result["documents"][0]:
    print("----------------")
    print(doc[:700])

print(result["metadatas"])

----------------

# Уход за ювелирными изделиями

## Общие правила ухода

Ювелирные украшения рассчитаны на длительное использование, однако их внешний вид во многом зависит от правильного ухода. Даже самые прочные драгоценные металлы и камни постепенно теряют блеск при постоянном контакте с косметикой, бытовой химией, потом и загрязнениями окружающей среды.

Регулярная очистка и правильное хранение позволяют сохранить первоначальный вид изделия на долгие годы.

Перед выполнением домашних работ, посещением бассейна, сауны, спортивных тренировок или использованием агрессивных чистящих средств рекомендуется снимать украшения. Это снижает риск механических повреждений и воздействия химических веществ на металл 
[[{'source': 'care.txt'}]]


Так же проверим несколько типичных вопросов клиента и посмотрим, какие документы были найдены:

In [24]:
questions = [
    "Почему серебро темнеет?",
    "Как выбрать кольцо для помолвки?",
    "Что такое проба золота?",
    "Как ухаживать за жемчугом?"
]

for q in questions:
    result = collection.query(
        query_texts=[q],
        n_results=1
    )

    print("\nВопрос:", q)
    print("Источник:", result["metadatas"][0][0]["source"])


Вопрос: Почему серебро темнеет?
Источник: gemstones.txt

Вопрос: Как выбрать кольцо для помолвки?
Источник: buying_guide.txt

Вопрос: Что такое проба золота?
Источник: gold_and_silver.txt

Вопрос: Как ухаживать за жемчугом?
Источник: buying_guide.txt


## Глава 3. Создание каталога товаров

### 3.1. Формирование каталога товаров

Для демонстрации работы виртуального консультанта создадим каталог ювелирных изделий.

Каждый товар содержит информацию, необходимую для семантического поиска:
- категория изделия;
- название;
- металл;
- вставка;
- описание;
- рекомендации по использованию;
- стоимость;
- ссылка на товар.

Каталог будет сохранён в формате JSON и позже загружен в отдельную коллекцию ChromaDB.

In [25]:
catalog = [

    {
        "id": 1,
        "category": "Кольцо",
        "name": "Кольцо «Аврора»",
        "metal": "Белое золото 585",
        "gemstone": "Бриллиант",
        "description": "Элегантное кольцо из белого золота 585 пробы с бриллиантом 0.25 карата. Классический дизайн делает его отличным выбором для помолвки и повседневного ношения.",
        "usage": "Помолвка, подарок, повседневная носка",
        "price": 68990,
        "url": "https://example.com/catalog/ring-aurora"
    },

    {
        "id": 2,
        "category": "Кольцо",
        "name": "Кольцо «Верона»",
        "metal": "Красное золото 585",
        "gemstone": "Рубин",
        "description": "Изящное кольцо с натуральным рубином насыщенного красного цвета. Подходит для торжественных мероприятий и станет выразительным подарком.",
        "usage": "Праздничные мероприятия, подарок",
        "price": 42990,
        "url": "https://example.com/catalog/ring-verona"
    },

    {
        "id": 3,
        "category": "Кольцо",
        "name": "Кольцо «Элегия»",
        "metal": "Серебро 925",
        "gemstone": "Фианит",
        "description": "Лаконичное серебряное кольцо с прозрачным фианитом. Отличный вариант для ежедневного использования и первого ювелирного украшения.",
        "usage": "Повседневная носка",
        "price": 6490,
        "url": "https://example.com/catalog/ring-elegy"
    },

    {
        "id": 4,
        "category": "Кольцо",
        "name": "Кольцо «Сапфир»",
        "metal": "Белое золото 750",
        "gemstone": "Сапфир",
        "description": "Премиальное кольцо с натуральным сапфиром овальной огранки и дорожкой из бриллиантов. Подходит для особых случаев.",
        "usage": "Особые случаи, подарок",
        "price": 118500,
        "url": "https://example.com/catalog/ring-sapphire"
    },

    {
        "id": 5,
        "category": "Серьги",
        "name": "Серьги «Классика»",
        "metal": "Белое золото 585",
        "gemstone": "Бриллиант",
        "description": "Классические серьги-пусеты с бриллиантами. Универсальная модель, которая сочетается практически с любым образом.",
        "usage": "Повседневная носка, деловой стиль",
        "price": 56990,
        "url": "https://example.com/catalog/earrings-classic"
    },

    {
        "id": 6,
        "category": "Серьги",
        "name": "Серьги «Созвездие»",
        "metal": "Красное золото 585",
        "gemstone": "Топаз",
        "description": "Легкие серьги с голубыми топазами. Подчеркивают элегантность и подходят как для офиса, так и для вечерних прогулок.",
        "usage": "Повседневная носка, деловой стиль",
        "price": 31990,
        "url": "https://example.com/catalog/earrings-constellation"
    },

    {
        "id": 7,
        "category": "Серьги",
        "name": "Серьги «Жемчужина»",
        "metal": "Серебро 925",
        "gemstone": "Натуральный жемчуг",
        "description": "Классические серебряные серьги с пресноводным жемчугом. Универсальная модель для любого возраста.",
        "usage": "Повседневная носка, торжественные мероприятия",
        "price": 14990,
        "url": "https://example.com/catalog/earrings-pearl"
    },

    {
        "id": 8,
        "category": "Серьги",
        "name": "Серьги «Лазурь»",
        "metal": "Белое золото 585",
        "gemstone": "Сапфир",
        "description": "Элегантные серьги с натуральными сапфирами глубокого синего цвета. Хорошо сочетаются с вечерними нарядами.",
        "usage": "Праздничные мероприятия",
        "price": 74990,
        "url": "https://example.com/catalog/earrings-lazur"
    },

    {
        "id": 9,
        "category": "Подвеска",
        "name": "Подвеска «Сердце»",
        "metal": "Красное золото 585",
        "gemstone": "Без вставок",
        "description": "Минималистичная подвеска в форме сердца. Подходит в качестве романтичного подарка.",
        "usage": "Повседневная носка, подарок",
        "price": 18990,
        "url": "https://example.com/catalog/pendant-heart"
    },

    {
        "id": 10,
        "category": "Подвеска",
        "name": "Подвеска «Капля»",
        "metal": "Белое золото 585",
        "gemstone": "Изумруд",
        "description": "Подвеска с натуральным изумрудом каплевидной формы. Изысканное украшение для торжественных мероприятий.",
        "usage": "Праздничные мероприятия",
        "price": 45990,
        "url": "https://example.com/catalog/pendant-drop"
    },

     {
        "id": 11,
        "category": "Подвеска",
        "name": "Подвеска «Виктория»",
        "metal": "Серебро 925",
        "gemstone": "Аметист",
        "description": "Серебряная подвеска с натуральным аметистом фиолетового оттенка. Элегантное украшение для повседневного образа.",
        "usage": "Повседневная носка, подарок",
        "price": 8990,
        "url": "https://example.com/catalog/pendant-victoria"
    },

    {
        "id": 12,
        "category": "Браслет",
        "name": "Браслет «Infinity»",
        "metal": "Белое золото 585",
        "gemstone": "Без вставок",
        "description": "Тонкий браслет из белого золота с символом бесконечности. Лаконичный дизайн подходит для ежедневного ношения.",
        "usage": "Повседневная носка, деловой стиль",
        "price": 38990,
        "url": "https://example.com/catalog/bracelet-infinity"
    },

    {
        "id": 13,
        "category": "Браслет",
        "name": "Браслет «Луна»",
        "metal": "Серебро 925",
        "gemstone": "Фианиты",
        "description": "Изящный серебряный браслет с декоративными вставками из фианитов. Легкое украшение для повседневных образов.",
        "usage": "Повседневная носка",
        "price": 11990,
        "url": "https://example.com/catalog/bracelet-luna"
    },

    {
        "id": 14,
        "category": "Браслет",
        "name": "Браслет «Classic»",
        "metal": "Красное золото 585",
        "gemstone": "Без вставок",
        "description": "Классический браслет из красного золота с гладкой полированной поверхностью. Универсальная модель для разных стилей.",
        "usage": "Повседневная носка, деловой стиль",
        "price": 62990,
        "url": "https://example.com/catalog/bracelet-classic"
    },

    {
        "id": 15,
        "category": "Цепочка",
        "name": "Цепочка «Венеция»",
        "metal": "Золото 585",
        "gemstone": "Без вставок",
        "description": "Классическая цепочка плетения «Венеция» из золота 585 пробы. Подходит для самостоятельного ношения и сочетания с подвесками.",
        "usage": "Повседневная носка",
        "price": 35990,
        "url": "https://example.com/catalog/chain-venice"
    },

    {
        "id": 16,
        "category": "Цепочка",
        "name": "Цепочка «Якорная»",
        "metal": "Серебро 925",
        "gemstone": "Без вставок",
        "description": "Прочная серебряная цепочка классического якорного плетения. Подходит для ежедневного использования.",
        "usage": "Повседневная носка",
        "price": 7990,
        "url": "https://example.com/catalog/chain-anchor"
    },

    {
        "id": 17,
        "category": "Колье",
        "name": "Колье «Северное сияние»",
        "metal": "Белое золото 585",
        "gemstone": "Бриллиант",
        "description": "Элегантное колье из белого золота с бриллиантовыми вставками. Украшение для торжественных случаев и особых событий.",
        "usage": "Торжественные мероприятия",
        "price": 124990,
        "url": "https://example.com/catalog/necklace-northern-light"
    },

    {
        "id": 18,
        "category": "Колье",
        "name": "Колье «Эстель»",
        "metal": "Серебро 925",
        "gemstone": "Жемчуг",
        "description": "Женственное колье с натуральным жемчугом и серебряной основой. Подчеркивает классический стиль.",
        "usage": "Праздничные мероприятия, подарок",
        "price": 26990,
        "url": "https://example.com/catalog/necklace-estelle"
    },

    {
        "id": 19,
        "category": "Брошь",
        "name": "Брошь «Лилия»",
        "metal": "Белое золото 585",
        "gemstone": "Изумруд",
        "description": "Изысканная брошь в форме цветка с натуральным изумрудом. Подходит для классического и вечернего образа.",
        "usage": "Торжественные мероприятия, коллекционное украшение",
        "price": 57990,
        "url": "https://example.com/catalog/brooch-lily"
    },

    {
        "id": 20,
        "category": "Часы",
        "name": "Часы «Prestige»",
        "metal": "Сталь с покрытием под золото",
        "gemstone": "Бриллианты",
        "description": "Элегантные женские часы с декоративными бриллиантовыми отметками на циферблате. Сочетают функциональность и ювелирный стиль.",
        "usage": "Деловой стиль, подарки",
        "price": 139990,
        "url": "https://example.com/catalog/watch-prestige"
    }

]

In [26]:
print(f"Всего товаров: {len(catalog)}")

Всего товаров: 20


In [27]:
for item in catalog:
    print(item["id"], item["name"])

1 Кольцо «Аврора»
2 Кольцо «Верона»
3 Кольцо «Элегия»
4 Кольцо «Сапфир»
5 Серьги «Классика»
6 Серьги «Созвездие»
7 Серьги «Жемчужина»
8 Серьги «Лазурь»
9 Подвеска «Сердце»
10 Подвеска «Капля»
11 Подвеска «Виктория»
12 Браслет «Infinity»
13 Браслет «Луна»
14 Браслет «Classic»
15 Цепочка «Венеция»
16 Цепочка «Якорная»
17 Колье «Северное сияние»
18 Колье «Эстель»
19 Брошь «Лилия»
20 Часы «Prestige»


### 3.2. Сохранение каталога в формате JSON

Каталог товаров сохраняется в отдельный JSON-файл.

Такой формат удобен для хранения структурированных данных и легко используется при последующей загрузке в ChromaDB.

Использование отдельного файла позволяет изменять ассортимент магазина без изменения основного кода программы.

In [28]:
import json

with open("data/catalog.json", "w", encoding="utf-8") as f:
    json.dump(
        catalog,
        f,
        ensure_ascii=False,
        indent=4
    )

print("Каталог из 20 товаров сохранён.")

Каталог из 20 товаров сохранён.


In [29]:
with open("data/catalog.json", encoding="utf-8") as f:
    loaded_catalog = json.load(f)

print(f"Загружено товаров: {len(loaded_catalog)}")

Загружено товаров: 20


In [30]:
from collections import Counter

categories = Counter(
    item["category"]
    for item in loaded_catalog
)

print(categories)

Counter({'Кольцо': 4, 'Серьги': 4, 'Подвеска': 3, 'Браслет': 3, 'Цепочка': 2, 'Колье': 2, 'Брошь': 1, 'Часы': 1})


## Глава 4. Векторизация каталога товаров

### 4.1. Загрузка каталога из JSON

На этом этапе загружаем каталог товаров из файла `catalog.json`.

Использование отдельного JSON-файла позволяет хранить ассортимент независимо от программного кода и при необходимости обновлять его без изменения логики приложения.

In [31]:
import json

with open("data/catalog.json", encoding="utf-8") as f:
    catalog = json.load(f)

print(f"Загружено товаров: {len(catalog)}")

Загружено товаров: 20


### 4.2. Подготовка товаров для ChromaDB

In [32]:
from langchain_core.documents import Document

product_documents = []

for item in catalog:

    text = f"""
Название: {item["name"]}

Категория: {item["category"]}

Металл: {item["metal"]}

Вставка: {item["gemstone"]}

Описание:
{item["description"]}

Рекомендуется для:
{item["usage"]}

Цена:
{item["price"]} рублей.
"""

    product_documents.append(
        Document(
            page_content=text,
            metadata=item
        )
    )

print(f"Подготовлено документов: {len(product_documents)}")

Подготовлено документов: 20


In [33]:
print(product_documents[0].page_content)
print("-" * 60)
print(product_documents[0].metadata)


Название: Кольцо «Аврора»

Категория: Кольцо

Металл: Белое золото 585

Вставка: Бриллиант

Описание:
Элегантное кольцо из белого золота 585 пробы с бриллиантом 0.25 карата. Классический дизайн делает его отличным выбором для помолвки и повседневного ношения.

Рекомендуется для:
Помолвка, подарок, повседневная носка

Цена:
68990 рублей.

------------------------------------------------------------
{'id': 1, 'category': 'Кольцо', 'name': 'Кольцо «Аврора»', 'metal': 'Белое золото 585', 'gemstone': 'Бриллиант', 'description': 'Элегантное кольцо из белого золота 585 пробы с бриллиантом 0.25 карата. Классический дизайн делает его отличным выбором для помолвки и повседневного ношения.', 'usage': 'Помолвка, подарок, повседневная носка', 'price': 68990, 'url': 'https://example.com/catalog/ring-aurora'}


### 4.3. Создание коллекции товаров

Для хранения эмбеддингов товаров создаётся отдельная коллекция ChromaDB.

Разделение базы знаний и каталога позволяет независимо выполнять поиск по информационным материалам и по ассортименту магазина.

In [34]:
products_collection = client.get_or_create_collection(
    name="products_catalog"
)

print("Коллекция товаров создана.")

Коллекция товаров создана.


### 4.4. Подготовка данных для загрузки

Перед добавлением товаров в ChromaDB подготавливаются отдельные списки идентификаторов, текстов документов и метаданных.

Такой формат соответствует требованиям метода `collection.add()`.

In [35]:
ids = [
    str(doc.metadata["id"])
    for doc in product_documents
]

texts = [
    doc.page_content
    for doc in product_documents
]

metadatas = [
    doc.metadata
    for doc in product_documents
]

print(f"Подготовлено документов: {len(texts)}")

Подготовлено документов: 20


In [36]:
print(f"Количество ID: {len(ids)}")
print(f"Количество текстов: {len(texts)}")
print(f"Количество metadata: {len(metadatas)}")

print("\nПервый ID:", ids[0])
print("\nНачало первого документа:")
print(texts[0][:250] + "...")

print("\nMetadata первого товара:")
print(metadatas[0])

Количество ID: 20
Количество текстов: 20
Количество metadata: 20

Первый ID: 1

Начало первого документа:

Название: Кольцо «Аврора»

Категория: Кольцо

Металл: Белое золото 585

Вставка: Бриллиант

Описание:
Элегантное кольцо из белого золота 585 пробы с бриллиантом 0.25 карата. Классический дизайн делает его отличным выбором для помолвки и повседневног...

Metadata первого товара:
{'id': 1, 'category': 'Кольцо', 'name': 'Кольцо «Аврора»', 'metal': 'Белое золото 585', 'gemstone': 'Бриллиант', 'description': 'Элегантное кольцо из белого золота 585 пробы с бриллиантом 0.25 карата. Классический дизайн делает его отличным выбором для помолвки и повседневного ношения.', 'usage': 'Помолвка, подарок, повседневная носка', 'price': 68990, 'url': 'https://example.com/catalog/ring-aurora'}


4.5. Загрузка товаров в ChromaDB

После подготовки идентификаторов, текстов и метаданных товары загружаются в коллекцию ChromaDB.

Каждый товар автоматически преобразуется в векторное представление, благодаря чему в дальнейшем его можно будет находить по смыслу, а не только по точному совпадению слов.

In [37]:
products_collection.upsert(
    ids=ids,
    documents=texts,
    metadatas=metadatas
)

print("Каталог товаров успешно сохранён в ChromaDB.")

Каталог товаров успешно сохранён в ChromaDB.


### 4.6. Тестирование поиска по каталогу

In [38]:
queries = [
    "Подберите кольцо для помолвки",
    "Хочу серебряные серьги",
    "Нужно украшение с жемчугом",
    "Подарок девушке до 30000 рублей"
]

for query in queries:
    print("=" * 60)
    print("Запрос:", query)

    result = products_collection.query(
        query_texts=[query],
        n_results=3
    )

    for item in result["metadatas"][0]:
        print(
            f'{item["name"]} | '
            f'{item["metal"]} | '
            f'{item["price"]} ₽'
        )

Запрос: Подберите кольцо для помолвки
Подвеска «Сердце» | Красное золото 585 | 18990 ₽
Кольцо «Аврора» | Белое золото 585 | 68990 ₽
Кольцо «Сапфир» | Белое золото 750 | 118500 ₽
Запрос: Хочу серебряные серьги
Серьги «Лазурь» | Белое золото 585 | 74990 ₽
Колье «Эстель» | Серебро 925 | 26990 ₽
Серьги «Жемчужина» | Серебро 925 | 14990 ₽
Запрос: Нужно украшение с жемчугом
Серьги «Жемчужина» | Серебро 925 | 14990 ₽
Колье «Эстель» | Серебро 925 | 26990 ₽
Серьги «Лазурь» | Белое золото 585 | 74990 ₽
Запрос: Подарок девушке до 30000 рублей
Кольцо «Сапфир» | Белое золото 750 | 118500 ₽
Подвеска «Сердце» | Красное золото 585 | 18990 ₽
Подвеска «Виктория» | Серебро 925 | 8990 ₽


Замечание.

Семантический поиск позволяет находить товары по смыслу запроса, однако не учитывает строгие числовые ограничения (например, диапазон цен) или точные условия фильтрации.

Для реализации таких возможностей обычно используется дополнительная фильтрация по метаданным или логика приложения после получения результатов поиска.

## Глава 5. Создание RAG-консультанта

### 5.1. Поиск информации в базе знаний

На первом этапе по запросу пользователя выполняется семантический поиск в базе знаний.

Найденные документы содержат справочную информацию о ювелирных изделиях, материалах, драгоценных камнях и правилах ухода.

В дальнейшем эти данные будут использоваться языковой моделью при формировании ответа.

In [39]:
query = "Как ухаживать за золотым кольцом с бриллиантом?"

knowledge_result = collection.query(
    query_texts=[query],
    n_results=2
)

for i, doc in enumerate(knowledge_result["documents"][0], 1):
    print(f"\nДокумент {i}")
    print("-" * 60)
    print(doc[:700])


Документ 1
------------------------------------------------------------

# Уход за ювелирными изделиями

## Общие правила ухода

Ювелирные украшения рассчитаны на длительное использование, однако их внешний вид во многом зависит от правильного ухода. Даже самые прочные драгоценные металлы и камни постепенно теряют блеск при постоянном контакте с косметикой, бытовой химией, потом и загрязнениями окружающей среды.

Регулярная очистка и правильное хранение позволяют сохранить первоначальный вид изделия на долгие годы.

Перед выполнением домашних работ, посещением бассейна, сауны, спортивных тренировок или использованием агрессивных чистящих средств рекомендуется снимать украшения. Это снижает риск механических повреждений и воздействия химических веществ на металл 

Документ 2
------------------------------------------------------------
# Золото и серебро в ювелирных изделиях

## Что такое проба

Проба показывает содержание драгоценного металла в сплаве. Чистые металлы часто слишком мягк

### 5.2. Поиск товаров по запросу

Помимо поиска справочной информации выполняется поиск наиболее подходящих товаров из каталога магазина.

В дальнейшем найденные товары будут использоваться языковой моделью при формировании персонализированной рекомендации пользователю.

In [40]:
query = "Мне нужен подарок девушке в виде подвески до 30000 рублей"


products_result = products_collection.query(
    query_texts=[query],
    n_results=5
)


filtered_products = []

for item in products_result["metadatas"][0]:

    # фильтр по бюджету
    if item["price"] > 30000:
        continue

    # фильтр по категории
    if "подвеск" in query.lower():
        if item["category"] != "Подвеска":
            continue

    filtered_products.append(item)


print("Подходящие товары:")

for item in filtered_products:
    print(f"{item['name']}")
    print(f"Категория: {item['category']}")
    print(f"Металл: {item['metal']}")
    print(f"Вставка: {item['gemstone']}")
    print(f"Цена: {item['price']} ₽")
    print("-" * 50)

Подходящие товары:
Подвеска «Сердце»
Категория: Подвеска
Металл: Красное золото 585
Вставка: Без вставок
Цена: 18990 ₽
--------------------------------------------------
Подвеска «Виктория»
Категория: Подвеска
Металл: Серебро 925
Вставка: Аметист
Цена: 8990 ₽
--------------------------------------------------


### 5.3. Формирование контекста для Llama

Результаты поиска из базы знаний и каталога товаров объединяются в единый контекст.

Этот контекст передаётся языковой модели Llama вместе с вопросом пользователя.

Модель использует только найденную информацию для формирования ответа консультанта.

In [41]:
context = """
=== БАЗА ЗНАНИЙ ===

"""

for doc in knowledge_result["documents"][0]:
    context += doc + "\n\n"


context += """
=== КАТАЛОГ ТОВАРОВ ===

"""

for product in filtered_products:
    context += f"""
Название: {product['name']}
Категория: {product['category']}
Металл: {product['metal']}
Вставка: {product['gemstone']}
Описание: {product['description']}
Цена: {product['price']} рублей

"""


print("Размер контекста:", len(context))

print("\n\n========== НАЧАЛО КОНТЕКСТА ==========\n")
print(context[:1500])

print("\n\n========== КОНЕЦ КОНТЕКСТА ==========\n")
print(context[-1500:])

Размер контекста: 8004


========== НАЧАЛО КОНТЕКСТА ==========


=== БАЗА ЗНАНИЙ ===


# Уход за ювелирными изделиями

## Общие правила ухода

Ювелирные украшения рассчитаны на длительное использование, однако их внешний вид во многом зависит от правильного ухода. Даже самые прочные драгоценные металлы и камни постепенно теряют блеск при постоянном контакте с косметикой, бытовой химией, потом и загрязнениями окружающей среды.

Регулярная очистка и правильное хранение позволяют сохранить первоначальный вид изделия на долгие годы.

Перед выполнением домашних работ, посещением бассейна, сауны, спортивных тренировок или использованием агрессивных чистящих средств рекомендуется снимать украшения. Это снижает риск механических повреждений и воздействия химических веществ на металл и вставки.

## Уход за золотыми украшениями

Золотые изделия рекомендуется очищать по мере загрязнения, обычно один раз в несколько месяцев.

Для домашней чистки подходит теплая вода с небольшим количеством жидког

### 5.4. Генерация ответа с помощью Llama

На этом этапе сформированный контекст и вопрос пользователя передаются языковой модели Llama.

Модель использует найденную информацию из базы знаний и каталога товаров для формирования ответа виртуального консультанта.

В отличие от обычного запроса к языковой модели, RAG позволяет получать ответы на основе данных конкретного магазина.

In [42]:
prompt = f"""
Ты — виртуальный консультант ювелирного магазина.

Отвечай только на основе предоставленного контекста.
Не придумывай товары, цены и характеристики, которых нет в данных.

Если пользователь указал бюджет, предлагай только товары,
которые соответствуют этому бюджету.
Если подходящих товаров нет, честно сообщи об этом.
Не предлагай товары с большей ценой.

Контекст:

{context}

Вопрос клиента:

{query}

Сформируй полезный и вежливый ответ для покупателя.
"""

response = client_llama.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2
)

answer = response.choices[0].message.content

print(answer)

Добрый день! Мы рады помочь вам выбрать подходящую подвеску для девушки.

Учитывая ваш бюджет в 30 000 рублей, я рекомендую следующие варианты:

1. Подвеска «Виктория» - серебряная подвеска с натуральным аметистом фиолетового оттенка. Цена: 8990 рублей.
2. Подвеска «Сердце» - минималистичная подвеска в форме сердца из красного золота 585. Цена: 18990 рублей.

Обе подвески являются красивыми и оригинальными вариантами, которые подойдут для романтического подарка. Если вы хотите выбрать более дешевую опцию, то подходит подвеска «Виктория». Если вы готовы потратить немного больше, то подходит подвеска «Сердце».

Если у вас есть какие-либо другие предпочтения или пожелания, я готов помочь вам найти идеальный подарок.


### 5.5. Финальный RAG-консультант

In [43]:
def ask_jewelry_assistant(query):

    # Поиск информации в базе знаний
    knowledge_result = collection.query(
        query_texts=[query],
        n_results=2
    )

    knowledge_docs = knowledge_result["documents"][0]


    # Поиск товаров в каталоге
    products_result = products_collection.query(
        query_texts=[query],
        n_results=3
    )

    product_docs = products_result["documents"][0]


    # Формируем общий контекст
    context = f"""
=== БАЗА ЗНАНИЙ ===

{chr(10).join(knowledge_docs)}


=== КАТАЛОГ ТОВАРОВ ===

{chr(10).join(product_docs)}
"""


    # Формируем запрос для Llama
    prompt = f"""
Ты — виртуальный консультант ювелирного магазина.

Отвечай только на основе предоставленного контекста.

Не придумывай товары, цены, характеристики и свойства,
которых нет в предоставленных данных.

Используй только информацию из каталога и базы знаний.
Не добавляй рекламные фразы, эмоциональные оценки
и описания от себя.

Работай как строгий консультант каталога.

Если пользователь ищет товар:
- предлагай только товары, которые есть в каталоге;
- сохраняй правильную категорию товара;
- если пользователь просит кольцо, предлагай только кольца;
- если пользователь просит подвеску, предлагай только подвески;
- не заменяй один вид украшения другим без просьбы пользователя.

Если пользователь указал бюджет:
- предлагай только товары, стоимость которых не превышает бюджет;
- товары выше бюджета полностью исключай из ответа;
- не упоминай товары дороже бюджета даже как альтернативу.

Если подходящих товаров нет:
- честно сообщи об отсутствии подходящих вариантов;
- не предлагай товары другой категории вместо них.

Если вопрос касается ухода или характеристик украшений:
- отвечай только на основе базы знаний;
- не добавляй информацию, которой нет в контексте.

Будь вежливым, кратким и полезным консультантом.


Контекст:

{context}


Вопрос клиента:

{query}


Сформируй ответ для покупателя.
"""


    # Генерация ответа
    response = client_llama.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )


    return response.choices[0].message.content

In [44]:
query = "Мне нужен подарок девушке в виде подвески до 30000 рублей"

answer = ask_jewelry_assistant(query)

print(answer)

Мы предлагаем две подвески, которые подходят для подарка и соответствуют вашему бюджету.

1. Подвеска «Сердце» из красного золота 585 пробы без вставок. Цена: 18990 рублей.
2. Подвеска «Виктория» из серебра 925 пробы с аметистом. Цена: 8990 рублей.

Обе подвески подойдут для повседневной носки и могут быть романтичным подарком.


## Глава 6. Создание интерактивного консультанта

### 6.1. Объединение компонентов RAG

На этом этапе все части системы объединяются в единую функцию:

- поиск информации в базе знаний;
- поиск товаров в каталоге;
- фильтрация результатов;
- формирование контекста;
- генерация ответа языковой моделью Llama.

In [45]:
def ask_jewelry_assistant(query):
    """
    Выполняет поиск информации и формирует ответ консультанта.
    """

    # поиск в базе знаний
    knowledge_result = collection.query(
        query_texts=[query],
        n_results=2
    )

    # поиск товаров
    products_result = products_collection.query(
        query_texts=[query],
        n_results=3
    )

    # фильтрация товаров
    filtered_products = []

    for item in products_result["metadatas"][0]:

        # фильтр по бюджету
        if "30000" in query:
            if item["price"] > 30000:
                continue

        # фильтр по категории
        if "подвеск" in query.lower():
            if item["category"] != "Подвеска":
                continue

        filtered_products.append(item)

    # формирование контекста
    context = """
=== БАЗА ЗНАНИЙ ===

"""

    for doc in knowledge_result["documents"][0]:
        context += doc + "\n\n"

    context += """
=== КАТАЛОГ ТОВАРОВ ===

"""

    for product in filtered_products:
        context += f"""
Название: {product['name']}
Категория: {product['category']}
Металл: {product['metal']}
Вставка: {product['gemstone']}
Описание: {product['description']}
Цена: {product['price']} рублей

"""

    # генерация ответа
    prompt = f"""
Ты — виртуальный консультант ювелирного магазина.

Отвечай только на основе предоставленного контекста.

Не придумывай товары, цены, характеристики и акции,
которых нет в контексте.

Если подходящих товаров нет, честно сообщи об этом.

Если вопрос касается ухода за украшениями,
используй информацию только из базы знаний.

Если вопрос касается выбора товара,
используй только товары из каталога.

Контекст:

{context}

Вопрос клиента:

{query}

Сформируй полезный, грамотный и вежливый ответ.
"""

    response = client_llama.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    answer = response.choices[0].message.content.strip()

    return answer

### 6.2. Тестирование консультанта

In [46]:
answer = ask_jewelry_assistant(
    "Мне нужен романтичный подарок девушке до 30000 рублей"
)

print(answer)

Девушка, я рад помочь вам выбрать романтичный подарок!

У нас есть несколько вариантов, которые подойдут для романтического подарка и не превышают 30 000 рублей.

Одним из вариантов может стать нашу подвеску "Сердце" из красного золота 585. Это минимальистичная подвеска в форме сердца, которая станет идеальным подарком для любимой. Цена этой подвески составляет 18 990 рублей.

Если вы ищете что-то еще, я могу предложить вам другие варианты. Например, мы можем рассмотреть серебряную цепочку или колье с драгоценным камнем. Однако, чтобы подобрать что-то идеальное, мне бы хотелось узнать больше о ваших предпочтениях и предпочтениях девушки.

Например, что вы думаете о цвете и стиле украшения? Имеет ли девушка какие-либо аллергические реакции или предпочтения по металлам? Это поможет мне предложить вам более точный и подходящий вариант.


In [47]:
answer = ask_jewelry_assistant(
    "Как ухаживать за золотым кольцом с бриллиантом?"
)

print(answer)

Добрый день! Я рад помочь вам в уходе за золотым кольцом с бриллиантом.

Чтобы сохранить первоначальный вид вашего украшения, мы рекомендуем следовать нескольким простым правилам:

1. Регулярно очищайте кольцо теплой водой с небольшим количеством жидкого мыла. Это поможет удалить загрязнения и сохранить блеск металла.
2. Избегайте использования жестких щеток, металлических губок, абразивных порошков и агрессивных химических веществ, которые могут оставить царапины на поверхности изделия.
3. Если кольцо покрыто родием, избегайте чрезмерной механической чистки, которая может постепенно удалить защитное покрытие. В таком случае рекомендуем обратиться к ювелиру для профессиональной полировки и повторного родирования.
4. Регулярно проверяйте крепление бриллианта, чтобы убедиться, что он надежно закреплен.
5. Храните кольцо отдельно в индивидуальном футляре или мягком мешочке, чтобы предотвратить появление царапин и механических повреждений.

Следуя этим простым правилам, вы сможете сохранит

In [48]:
answer = ask_jewelry_assistant(
    "Есть ли кольца с изумрудом до 5000 рублей?"
)

print(answer)

К сожалению, у нас нет кольцев с изумрудом в каталоге, и стоимость изумруда обычно достаточно высока, чтобы влить его в кольцо стоимостью до 5000 рублей. Однако я могу предложить вам другие варианты, которые могут вам понравиться.

Мы можем рассмотреть использование другого камня или дизайна, который подойдет к вашему бюджету. Кроме того, я могу подсказать вам, как подобрать кольцо с натуральным камнем, которое будет стоить меньше 5000 рублей.

Если вы хотите, я могу предложить вам несколько вариантов, которые могут вам понравиться.
